<a href="https://colab.research.google.com/github/SIRLLON/PB/blob/main/tp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 4 (TP4)
**Aluno:** Sirllon
**Plataforma:** Google Colab Notebook  
Implementa e testa Heaps Binárias, estruturas Trie, Grafos (BFS/DFS), Sockets TCP/UDP e diagnósticos de rede.

### Grupo 1: Heap Binária e Fila de Prioridades
Implementações de Heap Mínima, Heap Máxima e simulação de pouso de aviões.

In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def push(self, val):
        self.heap.append(val)
        self._up(len(self.heap)-1)

    def pop(self):
        if not self.heap: return None
        if len(self.heap) == 1: return self.heap.pop()
        res = self.heap[0]
        self.heap[0] = self.heap.pop()
        self._down(0)
        return res

    def _up(self, idx):
        while idx > 0:
            p = (idx - 1) // 2
            if self.heap[idx] < self.heap[p]:
                self.heap[idx], self.heap[p] = self.heap[p], self.heap[idx]
                idx = p
            else: break

    def _down(self, idx):
        n = len(self.heap)
        while 2 * idx + 1 < n:
            c = 2 * idx + 1
            if c + 1 < n and self.heap[c+1] < self.heap[c]: c += 1
            if self.heap[idx] > self.heap[c]:
                self.heap[idx], self.heap[c] = self.heap[c], self.heap[idx]
                idx = c
            else: break

def build_heap(arr):
    # Converte vetor em heap
    h = MinHeap()
    for x in arr: h.push(x)
    return h.heap

print("Build heap de [7, 5, 9, 1]:", build_heap([7, 5, 9, 1]))

#### Fila de Prioridade Pousos de Aviões
Fila de pouso priorizada por combustível (menor combustível pousa primeiro) e desempate por tempo.

In [ ]:
import heapq

class Aviao:
    def __init__(self, id_av, tempo_sol, combustivel):
        self.id = id_av
        self.tempo_sol = tempo_sol
        self.combustivel = combustivel

    def __lt__(self, other):
        # Prioridade por combustivel descrescente/crescente
        if self.combustivel == other.combustivel:
            return self.tempo_sol < other.tempo_sol
        return self.combustivel < other.combustivel

def simula_pousos(avioes):
    # Determina ordem de pousos
    heap = []
    for av in avioes:
        heapq.heappush(heap, av)

    sem_combustivel = 0
    tempo_atual = 0
    while heap:
        av = heapq.heappop(heap)
        if tempo_atual > av.tempo_sol + av.combustivel:
            sem_combustivel += 1
        # Simula pouso leva 1 tempo
        tempo_atual += 1
        print(f"Aviao {av.id} pousou no tempo {tempo_atual}")
    print(f"Qtd avioes sem combustivel: {sem_combustivel}")

### Grupo 2: Estrutura Trie de Palavras e Autocomplete

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False
        self.count = 0  # Palavras com o prefixo

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def inserir(self, word):
        # Insere palavra na Trie
        curr = self.root
        curr.count += 1
        for char in word:
            if char not in curr.children:
                curr.children[char] = TrieNode()
            curr = curr.children[char]
            curr.count += 1
        curr.is_word = True

    def autocomplete(self, prefix):
        # Autocomplete de palavras
        curr = self.root
        for char in prefix:
            if char not in curr.children: return []
            curr = curr.children[char]
        res = []
        self._dfs_words(curr, prefix, res)
        return res

    def _dfs_words(self, node, prefix, res):
        if node.is_word: res.append(prefix)
        for char, child in node.children.items():
            self._dfs_words(child, prefix + char, res)

t = Trie()
t.inserir("casa")
t.inserir("carro")
t.inserir("caminhao")
print("Autocomplete 'ca':", t.autocomplete("ca"))

### Grupo 3: Grafos e Busca BFS/DFS

In [ ]:
from collections import deque

class Grafo:
    def __init__(self):
        self.adj = {}

    def add_aresta(self, u, v):
        if u not in self.adj: self.adj[u] = []
        if v not in self.adj: self.adj[v] = []
        self.adj[u].append(v)
        self.adj[v].append(u)

    def bfs(self, start):
        # Busca em largura
        visited = set([start])
        queue = deque([start])
        order = []
        while queue:
            curr = queue.popleft()
            order.append(curr)
            for viz in self.adj.get(curr, []):
                if viz not in visited:
                    visited.add(viz)
                    queue.append(viz)
        return order

### Grupo 4 & 5: Sockets TCP/UDP e Diagnósticos de Rede
Implementamos servidores executados localmente em threads de background de forma funcional no Colab.

In [ ]:
import socket
import threading

def rodar_servidor_tcp():
    # Servidor simples TCP
    serv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    serv.bind(('127.0.0.1', 9991))
    serv.listen(1)
    conn, addr = serv.accept()
    msg = conn.recv(1024).decode()
    conn.send(f"ECO: {msg}".encode())
    conn.close()
    serv.close()

# Inicializa o servidor numa thread
t = threading.Thread(target=rodar_servidor_tcp)
t.start()

# Envia dados com o cliente
cli = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
cli.connect(('127.0.0.1', 9991))
cli.send(b"Ola Servidor!")
print("Resposta do servidor:", cli.recv(1024).decode())
cli.close()